In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

# Load directly from NYC Open Data (Socrata export endpoint preserves original column names)
ARRESTS_URL = 'https://data.cityofnewyork.us/api/views/uip8-fykc/rows.csv?accessType=DOWNLOAD'
CRASHES_URL = 'https://data.cityofnewyork.us/api/views/h9gi-nx95/rows.csv?accessType=DOWNLOAD'

arrests = pd.read_csv(ARRESTS_URL)
crashes = pd.read_csv(CRASHES_URL, low_memory=False)

print('Arrests shape:', arrests.shape)
print('Crashes shape:', crashes.shape)

In [2]:
print('   Arrests: Missing values    ')                                                                                      
print(arrests.isnull().sum()[arrests.isnull().sum() > 0])                                                                     
                                                                                                                                
print('\n   Crashes: Missing values    ')                                                                                    
print(crashes.isnull().sum()[crashes.isnull().sum() > 0])

   Arrests: Missing values    
KY_CD           23
LAW_CAT_CD    1474
dtype: int64

   Crashes: Missing values    
BOROUGH                           686249
ZIP CODE                          686532
LATITUDE                          240635
LONGITUDE                         240635
LOCATION                          240635
ON STREET NAME                    492348
CROSS STREET NAME                 860289
OFF STREET NAME                  1848051
NUMBER OF PERSONS INJURED             18
NUMBER OF PERSONS KILLED              31
CONTRIBUTING FACTOR VEHICLE 1       8104
CONTRIBUTING FACTOR VEHICLE 2     364149
CONTRIBUTING FACTOR VEHICLE 3    2085349
CONTRIBUTING FACTOR VEHICLE 4    2210833
CONTRIBUTING FACTOR VEHICLE 5    2237839
VEHICLE TYPE CODE 1                16733
VEHICLE TYPE CODE 2               456005
VEHICLE TYPE CODE 3              2091748
VEHICLE TYPE CODE 4              2212187
VEHICLE TYPE CODE 5              2238161
dtype: int64


In [3]:
# Arrests: drop rows missing critical fields                                                                                  
arrests.dropna(subset=['ARREST_DATE', 'ARREST_PRECINCT', 'Latitude', 'Longitude'], inplace=True)
                                                                                                                                
# Crashes: drop rows missing both location AND borough (can't do spatial analysis without either)
crashes.dropna(subset=['CRASH DATE', 'CRASH TIME'], inplace=True)

# Fill missing borough with 'UNKNOWN' rather than dropping because there would be too many to lose
crashes['BOROUGH'] = crashes['BOROUGH'].fillna('UNKNOWN')

print('Arrests after drop:', arrests.shape)
print('Crashes after drop:', crashes.shape)

Arrests after drop: (278953, 19)
Crashes after drop: (2248025, 29)


In [4]:
# Handle remaining missing values in arrests
arrests['LAW_CAT_CD'] = arrests['LAW_CAT_CD'].fillna('UNKNOWN')
arrests['KY_CD'] = arrests['KY_CD'].fillna(0)

# Handle missing injury counts in crashes — missing likely means 0 injuries
injury_cols = [
    'NUMBER OF PERSONS INJURED', 'NUMBER OF PERSONS KILLED',
    'NUMBER OF PEDESTRIANS INJURED', 'NUMBER OF PEDESTRIANS KILLED',
    'NUMBER OF CYCLIST INJURED', 'NUMBER OF CYCLIST KILLED',
    'NUMBER OF MOTORIST INJURED', 'NUMBER OF MOTORIST KILLED'
]
crashes[injury_cols] = crashes[injury_cols].fillna(0)

# Handle missing contributing factor — fill with UNKNOWN
crashes['CONTRIBUTING FACTOR VEHICLE 1'] = crashes['CONTRIBUTING FACTOR VEHICLE 1'].fillna('UNKNOWN')

print('Arrests LAW_CAT_CD missing after fix:', arrests['LAW_CAT_CD'].isnull().sum())
print('Arrests KY_CD missing after fix:', arrests['KY_CD'].isnull().sum())
print('Crashes injury cols missing after fix:', crashes[injury_cols].isnull().sum().sum())
print('Crashes CONTRIBUTING FACTOR VEHICLE 1 missing after fix:', crashes['CONTRIBUTING FACTOR VEHICLE 1'].isnull().sum())

Arrests LAW_CAT_CD missing after fix: 0
Arrests KY_CD missing after fix: 0
Crashes injury cols missing after fix: 0
Crashes CONTRIBUTING FACTOR VEHICLE 1 missing after fix: 0


In [5]:
# Arrests                                                                                                                     
arrests['ARREST_DATE'] = pd.to_datetime(arrests['ARREST_DATE'])                                                               
arrests['hour'] = arrests['ARREST_DATE'].dt.hour
arrests['day_of_week'] = arrests['ARREST_DATE'].dt.dayofweek  # 0=Monday
arrests['month'] = arrests['ARREST_DATE'].dt.month
arrests['year'] = arrests['ARREST_DATE'].dt.year

# Crashes
crashes['CRASH DATE'] = pd.to_datetime(crashes['CRASH DATE'])
crashes['CRASH DATETIME'] = pd.to_datetime(crashes['CRASH DATE'].astype(str) + ' ' + crashes['CRASH TIME'].astype(str), errors='coerce')
crashes['hour'] = crashes['CRASH DATETIME'].dt.hour
crashes['day_of_week'] = crashes['CRASH DATETIME'].dt.dayofweek
crashes['month'] = crashes['CRASH DATETIME'].dt.month
crashes['year'] = crashes['CRASH DATETIME'].dt.year

print(arrests[['ARREST_DATE', 'hour', 'day_of_week', 'month']].head(3))
print(crashes[['CRASH DATE', 'hour', 'day_of_week', 'month']].head(3))

  ARREST_DATE  hour  day_of_week  month
0  2025-01-10     0            4      1
1  2025-01-13     0            0      1
2  2025-01-13     0            0      1
  CRASH DATE  hour  day_of_week  month
0 2021-09-11     2            5      9
1 2022-03-26    11            5      3
2 2023-11-01     1            2     11


In [6]:
print('Arrests duplicates:', arrests.duplicated().sum())                                                                      
print('Crashes duplicates:', crashes.duplicated().sum())
                                                                                                                                
arrests.drop_duplicates(inplace=True)                     
crashes.drop_duplicates(inplace=True)

print('\nAfter deduplication:')
print('Arrests:', arrests.shape)
print('Crashes:', crashes.shape)

Arrests duplicates: 0
Crashes duplicates: 0

After deduplication:
Arrests: (278953, 23)
Crashes: (2248025, 34)


In [7]:
LAT_MIN, LAT_MAX = 40.4, 40.95                            
LON_MIN, LON_MAX = -74.3, -73.65

# Arrests
before = len(arrests)
arrests = arrests[(arrests['Latitude'].between(LAT_MIN, LAT_MAX)) & (arrests['Longitude'].between(LON_MIN, LON_MAX))]
print(f'Arrests: removed {before - len(arrests)} rows with invalid coordinates')

  # Crashes
before = len(crashes)
crashes = crashes[(crashes['LATITUDE'].between(LAT_MIN, LAT_MAX)) & (crashes['LONGITUDE'].between(LON_MIN, LON_MAX))]
print(f'Crashes: removed {before - len(crashes)} rows with invalid coordinates')

Arrests: removed 0 rows with invalid coordinates
Crashes: removed 247899 rows with invalid coordinates


In [8]:
crashes = crashes.copy()                                                                                                      
   
arrests['ARREST_BORO'] = arrests['ARREST_BORO'].str.strip().str.upper()                                                       
arrests['OFNS_DESC'] = arrests['OFNS_DESC'].str.strip().str.upper()
arrests['LAW_CAT_CD'] = arrests['LAW_CAT_CD'].str.strip().str.upper()

crashes['BOROUGH'] = crashes['BOROUGH'].str.strip().str.upper()
crashes['CONTRIBUTING FACTOR VEHICLE 1'] = crashes['CONTRIBUTING FACTOR VEHICLE 1'].str.strip().str.upper()

print('Arrests ARREST_BORO unique:', arrests['ARREST_BORO'].unique())
print('Crashes BOROUGH unique:', crashes['BOROUGH'].unique())

Arrests ARREST_BORO unique: ['Q' 'S' 'B' 'K' 'M']
Crashes BOROUGH unique: ['BROOKLYN' 'UNKNOWN' 'BRONX' 'MANHATTAN' 'QUEENS' 'STATEN ISLAND']


In [9]:
injury_cols = [                                                                                                               
'NUMBER OF PERSONS INJURED', 'NUMBER OF PERSONS KILLED',
'NUMBER OF PEDESTRIANS INJURED', 'NUMBER OF PEDESTRIANS KILLED',
'NUMBER OF CYCLIST INJURED', 'NUMBER OF CYCLIST KILLED',
'NUMBER OF MOTORIST INJURED', 'NUMBER OF MOTORIST KILLED']

for col in injury_cols:
    q99 = crashes[col].quantile(0.99)
    outliers = (crashes[col] > q99).sum()
    crashes[col] = crashes[col].clip(upper=q99)
    print(f'{col}: capped {outliers} outliers at {q99}')

NUMBER OF PERSONS INJURED: capped 14155 outliers at 3.0
NUMBER OF PERSONS KILLED: capped 2994 outliers at 0.0
NUMBER OF PEDESTRIANS INJURED: capped 4402 outliers at 1.0
NUMBER OF PEDESTRIANS KILLED: capped 1558 outliers at 0.0
NUMBER OF CYCLIST INJURED: capped 709 outliers at 1.0
NUMBER OF CYCLIST KILLED: capped 253 outliers at 0.0
NUMBER OF MOTORIST INJURED: capped 13810 outliers at 3.0
NUMBER OF MOTORIST KILLED: capped 1124 outliers at 0.0


In [ ]:
from pathlib import Path

Path('../data').mkdir(parents=True, exist_ok=True)

arrests.to_csv('../data/arrests_cleaned.csv', index=False)
crashes.to_csv('../data/crashes_cleaned.csv', index=False)

print('arrests_cleaned.csv saved:', arrests.shape)
print('crashes_cleaned.csv saved:', crashes.shape)